In [1]:
import pandas as pd
import time 
import requests
import json
#other requirements: sqlalchemy, psycopg2
#time & json are in python base 
#install requirements: pip install pandas requests sqlalchemy psycopg2-binary plyer

In [2]:
from DatabaseCreator import DatabaseCreator

In [3]:
start=time.perf_counter()

In [4]:
current_season = 20262027

In [5]:
from Load_Merge_FPL_Data import FPLDataManager
import CJDH_local_settings

#Run Database Creator
if __name__ == "__main__":
    manager = FPLDataManager(current_season=current_season)
    df_all_seasons = manager.load_all_seasons()
    print(f"\nTotal records: {len(df_all_seasons)}")

    db_creator = DatabaseCreator(db_settings=CJDH_local_settings.local_settings['FPL_Points_Predictor'])
    # fpl_engine = db_creator.get_engine_for("fpl_data_analysis") # not needed as we created the engine in the constructor of DatabaseCreator

    #Add a new staging table & data into the database
    playergw = df_all_seasons[['player_name_id','element','value','season','event','fixture','total_points',
                            'minutes','goals_scored','assists','team_elo','opp_team_elo','team_attack_elo', 
                            'opp_team_attack_elo', 'team_defence_elo', 'opp_team_defence_elo','position','bonus','bps',
                            'clean_sheets','goals_conceded','was_home','expected_assists','expected_goal_involvements',
                            'expected_goals','expected_goals_conceded','starts',
                            'cbi','defensive_contribution','recoveries','tackles',
                            'saves','team_name','opp_team_name']]

    table_name = "playergw"
    db_creator.create_staging_table_then_insert_data(table_name, data=playergw)
    playergwdf = db_creator.table_to_df(table_name=table_name)
    print("Final playergw dataframe loaded into psql!")
    print(playergwdf.info())

Loading season 2024-25 (manual merge)...
Loading season 2025-26...
Loading season 20262027 (API)...
Loaded data for element_id: 1/670
Loaded data for element_id: 2/670
Loaded data for element_id: 3/670
Loaded data for element_id: 4/670
Loaded data for element_id: 5/670
Loaded data for element_id: 6/670
Loaded data for element_id: 7/670
Loaded data for element_id: 8/670
Loaded data for element_id: 9/670
Loaded data for element_id: 10/670
Loaded data for element_id: 11/670
Loaded data for element_id: 12/670
Loaded data for element_id: 13/670
Loaded data for element_id: 14/670
Loaded data for element_id: 15/670
Loaded data for element_id: 16/670
Loaded data for element_id: 17/670
Loaded data for element_id: 18/670
Loaded data for element_id: 19/670
Loaded data for element_id: 20/670
Loaded data for element_id: 21/670
Loaded data for element_id: 22/670
Loaded data for element_id: 23/670
Loaded data for element_id: 24/670
Loaded data for element_id: 25/670
Loaded data for element_id: 26/670

In [6]:
filtered = playergwdf['player_name_id'] == "Jordan Pickford"
playergwdf[filtered].pivot_table(index='season',values='expected_goals',aggfunc='count')

,expected_goals
season,
20182019.0,0
20192020.0,0
20202021.0,0
20212022.0,0
20222023.0,38
20232024.0,38
20242025.0,38
20252026.0,38
20262027.0,9


In [7]:
filtered = (playergwdf['event']==1) & (playergwdf['player_name_id']=="Jordan Pickford")
playergwdf[filtered].sort_values(by='season')


,player_name_id,element,value,season,event,fixture,total_points,minutes,goals_scored,assists,...,expected_goals,expected_goals_conceded,starts,cbi,defensive_contribution,recoveries,tackles,saves,team_name,opp_team_name
267,Jordan Pickford,154.0,50.0,20182019.0,1.0,10.0,1.0,90.0,0.0,0.0,...,NaN,NaN,NaN,1.0,NaN,9.0,0.0,2.0,Everton,Wolves
22074,Jordan Pickford,148.0,55.0,20192020.0,1.0,4.0,7.0,90.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,Everton,Crystal Palace
44514,Jordan Pickford,157.0,50.0,20202021.0,1.0,4.0,8.0,90.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,Everton,Spurs
69153,Jordan Pickford,170.0,50.0,20212022.0,1.0,4.0,2.0,90.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,Everton,Southampton
94516,Jordan Pickford,182.0,45.0,20222023.0,1.0,3.0,3.0,90.0,0.0,0.0,...,0.0,0.00,0.0,NaN,NaN,NaN,NaN,5.0,Everton,Chelsea
121105,Jordan Pickford,263.0,45.0,20232024.0,1.0,5.0,2.0,90.0,0.0,0.0,...,0.0,1.50,1.0,NaN,NaN,NaN,NaN,1.0,Everton,Fulham
150785,Jordan Pickford,235.0,50.0,20242025.0,1.0,3.0,1.0,90.0,0.0,0.0,...,0.0,1.43,1.0,NaN,NaN,NaN,NaN,2.0,Everton,Brighton
178678,Jordan Pickford,287.0,55.0,20252026.0,1.0,10.0,2.0,90.0,0.0,0.0,...,0.0,2.07,1.0,3.0,0.0,8.0,0.0,2.0,Everton,Leeds
209962,Jordan Pickford,226.0,55.0,20262027.0,1.0,3.0,7.0,90.0,0.0,0.0,...,0.0,1.96,1.0,0.0,0.0,12.0,0.0,4.0,Everton,Crystal Palace


In [8]:
playergwdf.to_csv(r'data/playergwdf.csv',index=False)

# Get FPL Player current status

In [9]:
#Create DF from FPL API Request
headers={'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_10_1) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/39.0.2171.95 Safari/537.36'}
get=requests.get("https://fantasy.premierleague.com/api/bootstrap-static/", headers=headers, timeout=5)
data=json.loads(get.text)

#player_raw = Summary stats for each player for the current season
cols=list(data['elements'][0].keys())
player_raw = pd.DataFrame(data['elements'], columns = cols)
player = player_raw[['web_name','id','now_cost','status','news','chance_of_playing_next_round','chance_of_playing_this_round']]
player['season']=current_season
player['datetime_now'] = pd.to_datetime('now')

table_name = "player_status"
if __name__ == "__main__":
    # Replace the existing table entirely on each run
    db_creator.execute_sql(f"DROP TABLE IF EXISTS {table_name};")
    db_creator.create_staging_table_then_insert_data(table_name, data=player)
    player_status = db_creator.table_to_df(table_name=table_name)

In [10]:
view = db_creator.run_sql("""SELECT
player_name_id, 
SUM(expected_goals) AS total_expected_goals FROM playergw
WHERE season = 20262027
    AND expected_goals IS NOT NULL
    GROUP BY player_name_id
ORDER BY total_expected_goals DESC
LIMIT 10
                            """)
view

,player_name_id,total_expected_goals
0,Erling Haaland,6.56
1,Thierno Barry,5.08
2,Alexander Isak,4.27
3,Bryan Mbeumo,4.24
4,Igor Thiago Nascimento Rodrigues,4.06
5,Gonzalo García,3.97
6,Dominic Calvert-Lewin,3.91
7,Marcus Tavernier,3.71
8,Bukayo Saka,3.63
9,Francisco Evanilson de Lima Barbosa,3.60


In [11]:
finish=time.perf_counter()
print(f'Finished in {round(((finish-start)/60),2)} minute(s)')
#Finished in 11.91 minute(s)

Finished in 4.92 minute(s)


In [12]:
# Notification when file has finished running
from plyer import notification

notification.notify(
    title='Script Complete',
    message='Your file has finished running!',
    app_name='Jupyter Notebook',
    timeout=20  # notification stays for 20 seconds
)